# Docling's native document hierarchy vs the annotated text blob

The pipeline hands the model one string per document — the whole text, with
`**bold**`, `_italic_` and `++underline++` markers — and asks it to cut that string
into labelled pieces. That is the **annotated blob**. Docling produces something
richer-looking instead: a `DoclingDocument` with typed items (`section_header`,
`text`, `list_item`), heading levels, list groups, page and bounding-box provenance
for every item, and page headers/footers separated out as *furniture*.

The question here is whether that hierarchy carries the five DMP labels
(`title`, `section.title`, `section.description`, `question.text`, `answer.text`)
better than the blob does. Two halves:

1. **What the hierarchy actually contains** on the 10 samples — a census, and a
   side-by-side with the blob on two documents.
2. **How far structure alone gets** — the hierarchy mapped to the five labels by
   rules, with no model at all, scored with the pipeline's own evaluation beside
   the model runs. Built by `scripts/docling_structure_baseline.py`.

Same 10 documents, same 75% overlap threshold, both scoring paths, as everywhere
else in this project.


In [ ]:
import os
import sys
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
sys.path.insert(0, 'scripts')

import collections
import json
import logging
import re
import warnings

os.environ['TQDM_DISABLE'] = '1'
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

import pandas as pd
from IPython.display import display

from dmpbridge.core import paths as P
from dmpbridge.evaluation.annotation_rules import load_method_new
from dmpbridge.evaluation.evaluate import (
    LABELS, compute_f1_rows, load_method, micro_prf1,
)
from docling_structure_baseline import body_items, convert, pick_title

pd.set_option('display.max_colwidth', 72)
pd.set_option('display.width', 160)
SAMPLES = range(1, 11)
TAGS = {'Docling hierarchy, rules only': 'structure_docling_whole_doc', "Docling hierarchy, rules + ':'": 'structure-plus_docling_whole_doc', 'Docling native blob + gemma4:e4b': 'gemma4-e4b_docling_whole_doc', 'pdfplumber text blob + gemma4:e4b': 'gemma4-e4b_pdfplumber_whole_doc'}

docs = {n: convert(Path(f'data/input/pdfs/sample{n}.pdf')) for n in SAMPLES}
print('converted', len(docs), 'documents')


## 1. What the hierarchy contains

Every text-bearing item Docling produced, per document: its label, the heading
levels in use, the groups (lists, form areas) items are nested in, and what was
set aside as furniture.


In [ ]:
rows = []
for n, doc in docs.items():
    labels, levels, groups = collections.Counter(), collections.Counter(), collections.Counter()
    for item, depth in doc.iterate_items(with_groups=True):
        lab = str(getattr(item, 'label', ''))
        if hasattr(item, 'text'):
            labels[lab] += 1
            if lab == 'section_header':
                levels[item.level] += 1
        elif lab != 'unspecified':      # the root group
            groups[lab] += 1
    furniture = sum(1 for t in doc.texts if not str(t.content_layer).endswith('BODY'))
    rows.append({'sample': n, 'pages': len(doc.pages),
                 'section_header': labels['section_header'],
                 'header levels used': ','.join(str(k) for k in sorted(levels)),
                 'text': labels['text'], 'list_item': labels['list_item'],
                 'groups': ', '.join(f'{k}×{v}' for k, v in groups.items()) or '—',
                 'furniture': furniture, 'tables': len(doc.tables)})
census = pd.DataFrame(rows).set_index('sample')
display(census)
print('heading levels used anywhere in the corpus:',
      sorted({l for n in docs for it in docs[n].texts
              if str(it.label) == 'section_header' for l in [it.level]}))


**The hierarchy is flat.** Every `section_header` in all ten documents is level 1.
Sample 1's `Element 1: Data Type:` and the `A.`/`B.`/`C.` sub-headings beneath it
are siblings, not parent and children. The only nesting anywhere is list groups
(four documents) and one form area; no tables; furniture on two documents (a page
header on sample 6, a footer on sample 10). Nothing is labelled `title`.

### Side by side — sample 1

Docling's tree on the left, the pdfplumber blob the model actually receives on the
right. Same text, two representations.


In [ ]:
def tree(doc, limit=14):
    out = []
    for item, depth in list(doc.iterate_items(with_groups=True))[:limit + 1]:
        lab = str(getattr(item, 'label', ''))
        if lab == 'unspecified':
            continue
        txt = getattr(item, 'text', '')
        lvl = f' L{item.level}' if lab == 'section_header' else ''
        page = f' p{item.prov[0].page_no}' if getattr(item, 'prov', None) else ''
        out.append(f"{'  ' * (depth - 1)}{lab}{lvl}{page}: {txt[:52]!r}")
    return out


def blob(n, limit=14):
    text = json.loads(Path(f'data/output/1_extracted/pdfplumber/sample{n}.json')
                      .read_text(encoding='utf-8'))[0]['text']
    return [l[:66] for l in text.splitlines() if l.strip()][:limit]


def show(n):
    left, right = tree(docs[n]), blob(n)
    w = max(len(l) for l in left) + 2
    print(f"{'DOCLING HIERARCHY':<{w}}| PDFPLUMBER BLOB")
    print('-' * (w + 40))
    for a, b in zip(left + [''] * len(right), right + [''] * len(left)):
        if a or b:
            print(f'{a:<{w}}| {b}')

show(1)


Docling gives the blocks and their kinds, plus a page for each; the blob gives the
bold markers and one string. On sample 1 the two carry the same information: every
bold line is a Docling header, and vice versa — except the document title, which
Docling emits at the end of page 1 (its bbox is in the header region) while the
blob has it first.

### Side by side — sample 2


In [ ]:
show(2)


Here they diverge. The blob marks `** Roles & Responsibilities. **` as a bold
phrase at the start of a plain line, and marks the whole instruction paragraph
`_ A brief, high-level description… _` as italic. Docling has one `text` item for
`Roles & Responsibilities. For the proposed research…` — label and answer fused —
and the italic paragraph is another plain `text`. The hierarchy has no place to
put either fact.

### What each representation carries

| | pdfplumber blob | Docling hierarchy | matters for |
|---|---|---|---|
| block boundaries | no — the model cuts the string | yes, typed items | everything |
| heading vs body | bold marker (proxy) | `section_header` label | `section.title` |
| heading nesting | no | `level` — but always 1 here | sub-questions (sample 1) |
| lists | bare lines | list groups | multi-part answers |
| page / bbox | no | yes, every item | traceability; title position |
| headers/footers removed | no | furniture layer | noise |
| bold | yes, from font | no | inline labels → `question.text` |
| italic | yes, from font | no | `section.description` |
| underline | yes, from drawn rects | no | sample 6's headings |
| inline label split (`Label. Answer…`) | marker shows where | fused into one item | `question.text` |

Everything in the top half is structure Docling adds; everything in the bottom half
is typography only the blob has. Section 2 measures which half the labels depend on.


## 2. How far structure alone gets

`scripts/docling_structure_baseline.py` maps the hierarchy to the five labels with
no model and nothing read from the text:

| Docling item | becomes |
|---|---|
| the `section_header` highest on page 1 (by bbox) | `title` |
| every other `section_header` | `section.title` |
| `text`, `list_item` | `answer.text` |
| furniture | dropped |

`section.description` and `question.text` are never predicted — nothing in the
hierarchy distinguishes them from an answer. A second variant adds the one cue a
rule-based system would reach for: a short item ending in `:` or `?` becomes
`question.text`.

Both are scored exactly like a model run (stage 3 → Path A, stage 4 → Path B).


In [ ]:
def score(tag):
    df_a, conf_a, _ = load_method(tag, exclude=[])
    df_b, conf_b, _ = load_method_new(tag, exclude=[])
    m_a, m_b = micro_prf1(conf_a), micro_prf1(conf_b)
    return {'docs': len(df_a), 'precision': m_a['precision'], 'recall': m_a['recall'],
            'Path A': m_a['f1'], 'Path B': m_b['f1'],
            '_per_class': compute_f1_rows(conf_a).set_index('label'), '_docs': df_a}

S = {name: score(tag) for name, tag in TAGS.items()}
headline = pd.DataFrame({k: {c: v[c] for c in ('docs', 'precision', 'recall', 'Path A', 'Path B')}
                         for k, v in S.items()}).T
headline['docs'] = headline['docs'].astype(int)
display(headline.round(3))


Per class, Path A. Support is the number of gold items of that label across the
ten documents.


In [ ]:
pc = pd.concat({name: v['_per_class']['f1'] for name, v in S.items()}, axis=1)
pc['support'] = S[list(S)[0]]['_per_class']['support'].astype(int)
display(pc.loc[list(LABELS)].round(3))


Per document, Path A — the share of gold items each configuration labelled
correctly.


In [ ]:
pd_ = pd.concat({name: v['_docs'].set_index('sample')['accuracy']
                 for name, v in S.items()}, axis=1)
pd_.index = [int(s.replace('sample', '')) for s in pd_.index]
pd_ = pd_.sort_index()
display(pd_.round(3))
perfect = [n for n, v in pd_[list(S)[0]].items() if v == 1.0]
print('documents the rules-only hierarchy gets fully right:', perfect)


## 3. Where the hierarchy is right, and where it cannot be

**Right on half the corpus.** Five documents (4, 5, 7, 8, 9) are labelled perfectly
by the rules alone — no model, no fonts. These are plans whose structure is exactly
title → headings → answer paragraphs, and Docling's layout model recovers that
skeleton cleanly.

**Wrong in four ways, each visible in one document:**


In [ ]:
def gold(n):
    from dmpbridge.evaluation.evaluate import extract_gold, resolve_old_gt_path
    return extract_gold(resolve_old_gt_path(n))

def predicted(tag, n):
    return json.loads(P.labeled_path(tag, n).read_text(encoding='utf-8'))

tag = TAGS['Docling hierarchy, rules only']

print('(a) sample 1 — sub-headings are gold question.text, Docling section_header:')
g = {t: l for t, l in gold(1)}
for b in predicted(tag, 1)[:6]:
    print(f"    docling={b['docling_label']:<15} rule={b['label']:<14} gold={g.get(b['text'], '?'):<14} {b['text'][:48]!r}")
print()
print('(b) sample 2 — descriptions and inline labels, invisible to structure:')
for b in predicted(tag, 2)[2:5]:
    gl = next((l for t, l in gold(2) if t[:25] in b['text']), '?')
    print(f"    docling={b['docling_label']:<15} rule={b['label']:<14} gold≈{gl:<14} {b['text'][:48]!r}")
print()
print('(c) sample 6 — underlined headings fused into list items:')
for b in predicted(tag, 6)[:3]:
    print(f"    docling={b['docling_label']:<15} rule={b['label']:<14} {b['text'][:60]!r}")
print('    gold section.titles:', [t for t, l in gold(6) if l == 'section.title'][:3])
print()
print('(d) sample 10 — title by provenance picks the wrong header:')
t10 = pick_title(body_items(docs[10]))
print(f"    topmost header on page 1: {t10.text!r}   gold title: {[t for t, l in gold(10) if l == 'title']}")


- **(a) No second level.** Sample 1's `A.`/`B.`/`C.` items are questions in the
  annotation, but Docling makes them level-1 headers like `Element 1:`. Nine of the
  sixteen gold questions in the corpus are lost this way — the hierarchy has levels,
  but the layout model never used them on these documents.
- **(b) No typography.** `section.description` is italic instruction text; the inline
  `Label. Answer…` pattern is a bold phrase. The hierarchy fuses both into `text`.
  Docling reads the fonts into its page cells and never carries them up into the
  document — which is exactly what the `docling` extractor now reads directly
  (see `exploration-docling-native-format.ipynb`).
- **(c) Underline.** Sample 6's headings are underlined phrases at the start of list
  items; the layout model sees five list items, and the pdfplumber blob's
  `++ … ++` is the only representation that has them.
- **(d) Provenance is real but not decisive.** Using bbox to find the title fixed
  sample 1 (where reading order puts the title last) and broke sample 10 (where the
  topmost header is not the title). Nine of ten either way.


## 4. Conclusion

**The hierarchy is genuine but shallow on this corpus.** Docling's typed items are
a real advantage over a bare string — they give block boundaries and heading/body
for free, and that alone labels five of ten documents perfectly and scores 0.709
with no model at all. But the "rich" parts of the schema are empty here: every
heading is level 1, there are no tables, and the groups are four lists.

**The labels this task is scored on live in typography, not structure.** What
separates `question.text`, `section.description` and `answer.text` from each
other — and separates an inline label from its answer — is bold, italic and
underline. The hierarchy has none of the three. Docling's *native* page cells
have all three (fonts and hyperlinks), and the `docling` extractor now builds the
blob from them — the row in the tables above. That is why the ranking is what it is:

| representation | classifier | Path A |
|---|---|---|
| Docling hierarchy | rules | 0.709 |
| Docling Markdown export (headings only) as a blob — the earlier extractor, outputs backed up | gemma4:e4b | 0.767 |
| Docling native blob (fonts, links, headings) | gemma4:e4b | 0.924 |
| pdfplumber blob with all three markers | gemma4:e4b | **0.946** |

Structure alone gets 0.709. Structure plus a model reading Docling's Markdown got
0.767 — six points, because the model had nothing more to read than the rules did.
Structure plus the typography from Docling's own cells gets 0.924: the native
blob beats pdfplumber on samples 2 and 5, ties on seven, and loses only sample 6,
whose headings are drawn underlines that no level of Docling contains.

**Recommendation.** Keep the annotated blob as the representation the model
classifies — the hierarchy is a skeleton, not a substitute. Where the blob comes
from matters less than what it carries: pdfplumber's fonts and Docling's fonts
give almost the same markers, and the two-point gap between them is one document
with drawn underlines. What Docling's native output adds beyond the markers —
block boundaries, page and bounding-box provenance for every block, a confidence
per block, the furniture layer — is available for free once the cells are read,
and is the reason to prefer it when traceability matters.
